In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pytorch_lightning as pl

REPO_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "common" / "paths.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from common.paths import ROOT

from common.dataset import make_loader
from common.metrics import calculate_fid, dump_json, image_metric_frame, load_weight_only, regression_frames, save_weight_only, summarize_image_metrics, summarize_regression
from common.models import EmeanRegressor, PatchDiscriminator, TranslationNetwork, initialize_map2sat
from common.paths import PRETRAINED_GENERATOR
from common.split import assign_patient_split
from common.train import BestStateCallback, EpochRecorder

DATASET_NAME = "kidney"
NOTEBOOK_DIR = ROOT / "experiment" / "kidney" / "04_other"
BASE_PAIR_CSV = ROOT / "experiment" / "kidney" / "01_data_processing" / "cache" / "pairs_all.csv"
EXTERNAL_PAIR_CSV = ROOT / "experiment" / "kidney" / "01_data_processing" / "cache" / "external_pairs.csv"
training = False
SEEDS = [42, 43, 44, 45, 46]
BATCH_SIZE_GENERATOR = 8
BATCH_SIZE_EMEAN = 16
GENERATOR_EPOCHS = 80
EMEAN_EPOCHS = 60
NUM_WORKERS = 4

In [ ]:
class TranslationLightning(pl.LightningModule):
    def __init__(self, use_fem, learning_rate=2e-4, reconstruction_weight=100.0):
        super().__init__()
        self.automatic_optimization = False
        self.network = TranslationNetwork(use_fem=use_fem)
        self.discriminator = PatchDiscriminator()
        self.learning_rate = learning_rate
        self.reconstruction_weight = reconstruction_weight
        self.adversarial_loss = nn.BCEWithLogitsLoss()
        self.reconstruction_loss = nn.L1Loss()

    def forward(self, inputs):
        return self.network(inputs)

    def training_step(self, batch, batch_index):
        generator_optimizer, discriminator_optimizer = self.optimizers()
        gray = batch["gray"]
        target = batch["swe"]
        generated = self(gray)
        for parameter in self.discriminator.parameters():
            parameter.requires_grad_(True)
        real_logits = self.discriminator(gray, target)
        fake_logits = self.discriminator(gray, generated.detach())
        discriminator_loss = 0.5 * (
            self.adversarial_loss(real_logits, torch.ones_like(real_logits))
            + self.adversarial_loss(fake_logits, torch.zeros_like(fake_logits))
        )
        discriminator_optimizer.zero_grad()
        self.manual_backward(discriminator_loss)
        discriminator_optimizer.step()
        for parameter in self.discriminator.parameters():
            parameter.requires_grad_(False)
        fake_logits = self.discriminator(gray, generated)
        adversarial = self.adversarial_loss(fake_logits, torch.ones_like(fake_logits))
        reconstruction = self.reconstruction_loss(generated, target)
        generator_loss = adversarial + self.reconstruction_weight * reconstruction
        generator_optimizer.zero_grad()
        self.manual_backward(generator_loss)
        generator_optimizer.step()
        for parameter in self.discriminator.parameters():
            parameter.requires_grad_(True)
        self.log("train_generator_loss", generator_loss, prog_bar=True)
        self.log("train_discriminator_loss", discriminator_loss, prog_bar=True)
        self.log("train_reconstruction", reconstruction)
        return generator_loss

    def validation_step(self, batch, batch_index):
        generated = self(batch["gray"])
        loss = self.reconstruction_loss(generated, batch["swe"])
        self.log("val_reconstruction", loss, prog_bar=True, on_epoch=True)
        return loss

    def test_step(self, batch, batch_index):
        generated = self(batch["gray"])
        loss = self.reconstruction_loss(generated, batch["swe"])
        self.log("test_reconstruction", loss, on_epoch=True)
        return loss

    def predict_step(self, batch, batch_index, dataloader_index=0):
        return {
            "gray": batch["gray"].detach().cpu(),
            "prediction": self(batch["gray"]).detach().cpu(),
            "target": batch["swe"].detach().cpu(),
            "emean": batch["emean"].detach().cpu(),
            "image_name": list(batch["image_name"]),
            "patient_id": list(batch["patient_id"]),
        }

    def configure_optimizers(self):
        generator_optimizer = torch.optim.Adam(
            self.network.parameters(),
            lr=self.learning_rate,
            betas=(0.5, 0.999),
        )
        discriminator_optimizer = torch.optim.Adam(
            self.discriminator.parameters(),
            lr=self.learning_rate,
            betas=(0.5, 0.999),
        )
        return [generator_optimizer, discriminator_optimizer]

In [ ]:
class EmeanRegressionLightning(pl.LightningModule):
    def __init__(
        self,
        input_mode,
        target_mean,
        target_std,
        translation=None,
        learning_rate=1e-3,
    ):
        super().__init__()
        self.input_mode = input_mode
        self.translation = translation
        if self.translation is not None:
            self.translation.eval()
            for parameter in self.translation.parameters():
                parameter.requires_grad_(False)
        self.regressor = EmeanRegressor()
        self.register_buffer("target_mean", torch.tensor(float(target_mean)))
        self.register_buffer("target_std", torch.tensor(float(target_std)))
        self.learning_rate = learning_rate
        self.loss_function = nn.HuberLoss(delta=1.0)

    def on_train_epoch_start(self):
        if self.translation is not None:
            self.translation.eval()

    def _images(self, batch):
        if self.input_mode == "gray":
            return batch["gray"]
        if self.input_mode == "real_swe":
            return batch["swe"]
        if self.input_mode == "virtual_swe":
            with torch.no_grad():
                return self.translation(batch["gray"])
        raise ValueError(self.input_mode)

    def _standardized_prediction(self, batch):
        return self.regressor(self._images(batch))

    def forward(self, batch):
        standardized = self._standardized_prediction(batch)
        return torch.expm1(standardized * self.target_std + self.target_mean).clamp_min(0.0)

    def _loss(self, batch):
        target = (torch.log1p(batch["emean"]) - self.target_mean) / self.target_std
        return self.loss_function(self._standardized_prediction(batch), target)

    def training_step(self, batch, batch_index):
        loss = self._loss(batch)
        self.log("train_loss", loss, prog_bar=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_index):
        loss = self._loss(batch)
        self.log("val_loss", loss, prog_bar=True, on_epoch=True)
        return loss

    def test_step(self, batch, batch_index):
        loss = self._loss(batch)
        prediction = self(batch)
        mae = torch.mean(torch.abs(prediction - batch["emean"]))
        self.log("test_loss", loss, on_epoch=True)
        self.log("test_mae", mae, on_epoch=True)
        return loss

    def predict_step(self, batch, batch_index, dataloader_index=0):
        return {
            "prediction": self(batch).detach().cpu(),
            "target": batch["emean"].detach().cpu(),
            "image_name": list(batch["image_name"]),
            "patient_id": list(batch["patient_id"]),
        }

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.regressor.parameters(),
            lr=self.learning_rate,
            weight_decay=1e-4,
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=6,
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"},
        }

In [ ]:
def collect_translation(outputs):
    predictions = torch.cat([output["prediction"] for output in outputs])
    targets = torch.cat([output["target"] for output in outputs])
    image_names = [name for output in outputs for name in output["image_name"]]
    patient_ids = [name for output in outputs for name in output["patient_id"]]
    return predictions, targets, image_names, patient_ids


def collect_regression(outputs):
    predictions = torch.cat([output["prediction"] for output in outputs]).numpy()
    targets = torch.cat([output["target"] for output in outputs]).numpy()
    image_names = [name for output in outputs for name in output["image_name"]]
    patient_ids = [name for output in outputs for name in output["patient_id"]]
    return predictions, targets, image_names, patient_ids


base_frame = pd.read_csv(BASE_PAIR_CSV).drop(columns=["split"])
external_frame = pd.read_csv(EXTERNAL_PAIR_CSV)
rows = []
for seed in SEEDS:
    pl.seed_everything(seed, workers=True)
    frame = assign_patient_split(base_frame, seed=seed)
    frame.to_csv(NOTEBOOK_DIR / f"pairs_seed_{seed}.csv", index=False)
    train_frame = frame.loc[frame["split"] == "train"].copy()
    validation_frame = frame.loc[frame["split"] == "val"].copy()
    test_frame = frame.loc[frame["split"] == "test"].copy()
    train_loader = make_loader(train_frame, BATCH_SIZE_GENERATOR, True, True, NUM_WORKERS)
    validation_loader = make_loader(validation_frame, BATCH_SIZE_GENERATOR, False, False, NUM_WORKERS)
    test_loader = make_loader(test_frame, BATCH_SIZE_GENERATOR, False, False, NUM_WORKERS)
    external_loader = make_loader(external_frame, BATCH_SIZE_GENERATOR, False, False, NUM_WORKERS)
    generator_weight = NOTEBOOK_DIR / f"seed_{seed}_generator.pt"
    emean_weight = NOTEBOOK_DIR / f"seed_{seed}_emean_head.pt"
    translation_model = TranslationLightning(True, 2e-4)
    if training:
        initialize_map2sat(translation_model.network.generator, PRETRAINED_GENERATOR)
    generator_best = BestStateCallback("val_reconstruction", "network")
    generator_trainer = pl.Trainer(
        max_epochs=GENERATOR_EPOCHS,
        accelerator="auto",
        devices=1,
        precision="16-mixed" if torch.cuda.is_available() else "32-true",
        deterministic=True,
        logger=False,
        enable_checkpointing=False,
        enable_model_summary=False,
        enable_progress_bar=False,
        callbacks=[
            EpochRecorder(NOTEBOOK_DIR / f"seed_{seed}_generator_epoch_metrics.csv"),
            generator_best,
            pl.callbacks.EarlyStopping(
                monitor="val_reconstruction",
                mode="min",
                patience=15,
            ),
        ],
        num_sanity_val_steps=0,
    )
    if training:
        generator_trainer.fit(translation_model, train_loader, validation_loader)
        save_weight_only(translation_model.network, generator_weight)
        reloaded_network = TranslationNetwork(True)
        load_weight_only(reloaded_network, generator_weight)
        translation_model.network.load_state_dict(reloaded_network.state_dict())
    else:
        if not generator_weight.is_file():
            raise FileNotFoundError(generator_weight)
        load_weight_only(translation_model.network, generator_weight)
    generator_trainer.validate(translation_model, validation_loader)
    generator_trainer.test(translation_model, test_loader)
    translation_outputs = generator_trainer.predict(translation_model, test_loader)
    predictions, targets, image_names, patient_ids = collect_translation(translation_outputs)
    image_frame = image_metric_frame(predictions, targets, image_names, patient_ids)
    image_summary, patient_image_frame = summarize_image_metrics(image_frame)
    internal_fid = calculate_fid(
        predictions,
        targets,
        NOTEBOOK_DIR / f"fid_seed_{seed}_internal",
        "cuda:0" if torch.cuda.is_available() else "cpu",
    )
    image_frame.to_csv(NOTEBOOK_DIR / f"seed_{seed}_image_metrics.csv", index=False)
    patient_image_frame.to_csv(
        NOTEBOOK_DIR / f"seed_{seed}_patient_image_metrics.csv",
        index=False,
    )
    generator_trainer.test(translation_model, external_loader)
    external_translation_outputs = generator_trainer.predict(
        translation_model,
        external_loader,
    )
    external_predictions, external_targets, external_names, external_patients = collect_translation(
        external_translation_outputs
    )
    external_image_frame = image_metric_frame(
        external_predictions,
        external_targets,
        external_names,
        external_patients,
    )
    external_image_summary, external_patient_image_frame = summarize_image_metrics(
        external_image_frame
    )
    external_fid = calculate_fid(
        external_predictions,
        external_targets,
        NOTEBOOK_DIR / f"fid_seed_{seed}_external",
        "cuda:0" if torch.cuda.is_available() else "cpu",
    )
    external_image_frame.to_csv(
        NOTEBOOK_DIR / f"seed_{seed}_external_image_metrics.csv",
        index=False,
    )
    external_patient_image_frame.to_csv(
        NOTEBOOK_DIR / f"seed_{seed}_external_patient_image_metrics.csv",
        index=False,
    )
    log_targets = np.log1p(train_frame["emean"].to_numpy(dtype=float))
    target_mean = float(log_targets.mean())
    target_std = float(log_targets.std(ddof=1))
    if not np.isfinite(target_std) or target_std <= 0:
        raise ValueError("Invalid Emean standard deviation")
    train_emean_loader = make_loader(train_frame, BATCH_SIZE_EMEAN, True, True, NUM_WORKERS)
    validation_emean_loader = make_loader(validation_frame, BATCH_SIZE_EMEAN, False, False, NUM_WORKERS)
    test_emean_loader = make_loader(test_frame, BATCH_SIZE_EMEAN, False, False, NUM_WORKERS)
    external_emean_loader = make_loader(external_frame, BATCH_SIZE_EMEAN, False, False, NUM_WORKERS)
    emean_model = EmeanRegressionLightning(
        "virtual_swe",
        target_mean,
        target_std,
        translation_model.network,
        1e-3,
    )
    emean_best = BestStateCallback("val_loss", "regressor")
    emean_trainer = pl.Trainer(
        max_epochs=EMEAN_EPOCHS,
        accelerator="auto",
        devices=1,
        precision="16-mixed" if torch.cuda.is_available() else "32-true",
        deterministic=True,
        logger=False,
        enable_checkpointing=False,
        enable_model_summary=False,
        enable_progress_bar=False,
        callbacks=[
            EpochRecorder(NOTEBOOK_DIR / f"seed_{seed}_emean_epoch_metrics.csv"),
            emean_best,
            pl.callbacks.EarlyStopping(
                monitor="val_loss",
                mode="min",
                patience=12,
            ),
        ],
        num_sanity_val_steps=0,
    )
    if training:
        emean_trainer.fit(emean_model, train_emean_loader, validation_emean_loader)
        save_weight_only(emean_model.regressor, emean_weight)
        reloaded_regressor = EmeanRegressor()
        load_weight_only(reloaded_regressor, emean_weight)
        emean_model.regressor.load_state_dict(reloaded_regressor.state_dict())
    else:
        if not emean_weight.is_file():
            raise FileNotFoundError(emean_weight)
        load_weight_only(emean_model.regressor, emean_weight)
    emean_trainer.validate(emean_model, validation_emean_loader)
    emean_trainer.test(emean_model, test_emean_loader)
    regression_outputs = emean_trainer.predict(emean_model, test_emean_loader)
    emean_predictions, emean_targets, emean_names, emean_patients = collect_regression(regression_outputs)
    image_regression, patient_regression = regression_frames(
        emean_predictions,
        emean_targets,
        emean_names,
        emean_patients,
    )
    image_regression.to_csv(NOTEBOOK_DIR / f"seed_{seed}_predictions.csv", index=False)
    patient_regression.to_csv(
        NOTEBOOK_DIR / f"seed_{seed}_patient_predictions.csv",
        index=False,
    )
    regression_summary = summarize_regression(image_regression, patient_regression)
    emean_trainer.test(emean_model, external_emean_loader)
    external_regression_outputs = emean_trainer.predict(
        emean_model,
        external_emean_loader,
    )
    external_emean_predictions, external_emean_targets, external_emean_names, external_emean_patients = collect_regression(
        external_regression_outputs
    )
    external_image_regression, external_patient_regression = regression_frames(
        external_emean_predictions,
        external_emean_targets,
        external_emean_names,
        external_emean_patients,
    )
    external_image_regression.to_csv(
        NOTEBOOK_DIR / f"seed_{seed}_external_predictions.csv",
        index=False,
    )
    external_patient_regression.to_csv(
        NOTEBOOK_DIR / f"seed_{seed}_external_patient_predictions.csv",
        index=False,
    )
    external_regression_summary = summarize_regression(
        external_image_regression,
        external_patient_regression,
    )
    row = {
        "seed": seed,
        "image_internal_fid": internal_fid,
        "image_external_fid": external_fid,
    }
    for level, values in image_summary.items():
        for metric, statistics in values.items():
            row[f"image_internal_{level}_{metric}"] = statistics["mean"]
    for level, values in external_image_summary.items():
        for metric, statistics in values.items():
            row[f"image_external_{level}_{metric}"] = statistics["mean"]
    for level, values in regression_summary.items():
        for metric, value in values.items():
            row[f"emean_internal_{level}_{metric}"] = value
    for level, values in external_regression_summary.items():
        for metric, value in values.items():
            row[f"emean_external_{level}_{metric}"] = value
    rows.append(row)

seed_frame = pd.DataFrame(rows)
seed_frame.to_csv(NOTEBOOK_DIR / "five_seed_metrics.csv", index=False)
summary_rows = []
for column in seed_frame.columns:
    if column == "seed":
        continue
    values = pd.to_numeric(seed_frame[column], errors="coerce").dropna()
    summary_rows.append(
        {
            "metric": column,
            "mean": float(values.mean()),
            "sd": float(values.std(ddof=1)) if len(values) > 1 else 0.0,
        }
    )
summary_frame = pd.DataFrame(summary_rows)
summary_frame.to_csv(NOTEBOOK_DIR / "five_seed_mean_sd.csv", index=False)
dump_json(
    {"dataset": DATASET_NAME, "experiment": "five_seed_repeats", "results": summary_rows},
    NOTEBOOK_DIR / "metrics_five_seed_repeats.json",
)
display(summary_frame)